# Feature Engineering: Points Averages and Chronological Split

This notebook loads `data/sample/sample_player_game_logs.csv`, adds leakage-safe points features, and labels an 80/20 chronological train/test split. It does **not** write a new CSV unless `SAVE_OUTPUT` is set to `True` in the last section.

## Load the game logs

`game_id` is read as a string to keep its leading zeros. The source file is stored newest-first, so rows are sorted oldest-first before any rolling calculation.

In [1]:
import math
from pathlib import Path
import pandas as pd

# Locate the CSV from the notebook's folder or any parent (repo root).
DATA_RELATIVE_PATH = Path('data') / 'sample' / 'sample_player_game_logs.csv'
DATA_PATH = next(
    (root / DATA_RELATIVE_PATH for root in (Path.cwd(), *Path.cwd().parents) if (root / DATA_RELATIVE_PATH).exists()),
    None,
)
assert DATA_PATH is not None, f'Could not find {DATA_RELATIVE_PATH}'

games = pd.read_csv(DATA_PATH, dtype={'game_id': str}, parse_dates=['game_date'])
games = games.sort_values(['player_id', 'game_date']).reset_index(drop=True)
print(DATA_PATH)
print(games.shape)
games.head()

/Users/swayamshree/Desktop/Capstone Project/CPSC491-02Group8SportsBetting/data/sample/sample_player_game_logs.csv
(43, 25)


,player_id,game_id,game_date,matchup,win_loss,minutes,field_goals_made,field_goals_attempted,field_goal_pct,three_pointers_made,...,offensive_rebounds,defensive_rebounds,rebounds,assists,steals,blocks,turnovers,personal_fouls,points,plus_minus
0,201939,0022500002,2025-10-21,GSW @ LAL,W,32,6,14,0.429,3,...,0,1,1,4,3,1,2,3,23,1
1,201939,0022500006,2025-10-23,GSW vs. DEN,W,37,14,25,0.560,6,...,0,6,6,7,3,1,2,1,42,15
2,201939,0022500097,2025-10-24,GSW @ POR,L,27,12,22,0.545,7,...,1,5,6,3,1,2,4,1,35,-13
3,201939,0022500121,2025-10-27,GSW vs. MEM,W,30,5,11,0.455,4,...,0,4,4,3,0,0,1,2,16,17
4,201939,0022500127,2025-10-28,GSW vs. LAC,W,26,7,15,0.467,2,...,0,2,2,8,2,0,2,1,19,16


## Season key

NBA seasons run October through June, so a game in October or later belongs to that year's season and anything earlier belongs to the previous year's. This makes the season average reset correctly once more seasons are added.

In [2]:
games['season'] = games['game_date'].dt.year.where(
    games['game_date'].dt.month >= 10,
    games['game_date'].dt.year - 1,
)
games['season'].value_counts()

season
2025    43
Name: count, dtype: int64

## Points features

Each feature only uses games played **before** the current row (`shift(1)`), so a game's own points never leak into its features. A player's first game has no history, so its features are `NaN`; early games use partial windows.

- `pts_season_avg`: average points over prior games this season
- `pts_last5_avg`: average points over the prior 5 games
- `pts_last10_avg`: average points over the prior 10 games

In [3]:
prior_points = games.groupby('player_id')['points'].shift(1)
prior_season_points = games.groupby(['player_id', 'season'])['points'].shift(1)

games['pts_season_avg'] = prior_season_points.groupby(
    [games['player_id'], games['season']]
).transform(lambda s: s.expanding(min_periods=1).mean())

for window in (5, 10):
    games[f'pts_last{window}_avg'] = prior_points.groupby(games['player_id']).transform(
        lambda s: s.rolling(window, min_periods=1).mean()
    )

FEATURE_COLS = ['pts_season_avg', 'pts_last5_avg', 'pts_last10_avg']
games[FEATURE_COLS] = games[FEATURE_COLS].round(2)
games[['game_date', 'points', *FEATURE_COLS]].head(12)

,game_date,points,pts_season_avg,pts_last5_avg,pts_last10_avg
0,2025-10-21,23,NaN,NaN,NaN
1,2025-10-23,42,23.00,23.00,23.00
2,2025-10-24,35,32.50,32.50,32.50
3,2025-10-27,16,33.33,33.33,33.33
4,2025-10-28,19,29.00,29.00,29.00
5,2025-10-30,27,27.00,27.00,27.00
6,2025-11-01,24,27.00,27.80,27.00
7,2025-11-04,28,26.57,24.20,26.57
8,2025-11-11,11,26.75,22.80,26.75
9,2025-11-12,46,25.00,21.80,25.00


In [4]:
# Validation: recompute every feature from strictly earlier games.
for _, player_games in games.groupby('player_id'):
    points = player_games['points'].tolist()
    rows = player_games.reset_index(drop=True)
    assert rows.loc[0, FEATURE_COLS].isna().all()
    for i in range(1, len(rows)):
        prev = points[:i]
        assert rows.loc[i, 'pts_season_avg'] == round(sum(prev) / len(prev), 2)
        assert rows.loc[i, 'pts_last5_avg'] == round(sum(prev[-5:]) / len(prev[-5:]), 2)
        assert rows.loc[i, 'pts_last10_avg'] == round(sum(prev[-10:]) / len(prev[-10:]), 2)
print('All features use prior games only.')

All features use prior games only.


## Chronological 80/20 train/test split

The oldest 80% of game dates go to `train` and the newest 20% to `test`. Splitting on dates, not shuffled rows, keeps the model from training on games that happen after the ones it is tested on, and keeps every game day entirely in one set.

In [5]:
TRAIN_FRACTION = 0.8

game_dates = sorted(games['game_date'].unique())
train_cutoff = game_dates[math.floor(len(game_dates) * TRAIN_FRACTION) - 1]
games['split'] = (games['game_date'] <= train_cutoff).map({True: 'train', False: 'test'})

train = games[games['split'] == 'train']
test = games[games['split'] == 'test']
assert train['game_date'].max() < test['game_date'].min()

print(f"train: {len(train)} rows, {train['game_date'].min().date()} to {train['game_date'].max().date()}")
print(f"test:  {len(test)} rows, {test['game_date'].min().date()} to {test['game_date'].max().date()}")

train: 34 rows, 2025-10-21 to 2026-01-19
test:  9 rows, 2026-01-20 to 2026-04-12


## Save the new dataset (optional)

Set `SAVE_OUTPUT = True` to write the feature dataset next to the source CSV. It is off by default so running the notebook does not create files.

In [6]:
SAVE_OUTPUT = True
OUTPUT_PATH = DATA_PATH.with_name('sample_player_game_logs_features.csv')

output = games.sort_values(['game_date', 'player_id']).reset_index(drop=True)
output['game_date'] = output['game_date'].dt.strftime('%Y-%m-%d')

if SAVE_OUTPUT:
    output.to_csv(OUTPUT_PATH, index=False)
    print(f'Wrote {len(output)} rows to {OUTPUT_PATH}')
else:
    print(f'SAVE_OUTPUT is False; would write {len(output)} rows to {OUTPUT_PATH}')

Wrote 43 rows to /Users/swayamshree/Desktop/Capstone Project/CPSC491-02Group8SportsBetting/data/sample/sample_player_game_logs_features.csv


## Load new dataset

In [7]:
features = pd.read_csv(OUTPUT_PATH, dtype={'game_id': str}, parse_dates=['game_date'])
print(features.shape)
features.head(10)

(43, 30)


,player_id,game_id,game_date,matchup,win_loss,minutes,field_goals_made,field_goals_attempted,field_goal_pct,three_pointers_made,...,blocks,turnovers,personal_fouls,points,plus_minus,season,pts_season_avg,pts_last5_avg,pts_last10_avg,split
0,201939,0022500002,2025-10-21,GSW @ LAL,W,32,6,14,0.429,3,...,1,2,3,23,1,2025,NaN,NaN,NaN,train
1,201939,0022500006,2025-10-23,GSW vs. DEN,W,37,14,25,0.560,6,...,1,2,1,42,15,2025,23.00,23.00,23.00,train
2,201939,0022500097,2025-10-24,GSW @ POR,L,27,12,22,0.545,7,...,2,4,1,35,-13,2025,32.50,32.50,32.50,train
3,201939,0022500121,2025-10-27,GSW vs. MEM,W,30,5,11,0.455,4,...,0,1,2,16,17,2025,33.33,33.33,33.33,train
4,201939,0022500127,2025-10-28,GSW vs. LAC,W,26,7,15,0.467,2,...,0,2,1,19,16,2025,29.00,29.00,29.00,train
5,201939,0022500139,2025-10-30,GSW @ MIL,L,35,8,19,0.421,4,...,0,5,2,27,-6,2025,27.00,27.00,27.00,train
6,201939,0022500144,2025-11-01,GSW @ IND,L,29,8,23,0.348,4,...,0,5,2,24,-21,2025,27.00,27.80,27.00,train
7,201939,0022500169,2025-11-04,GSW vs. PHX,W,34,9,23,0.391,5,...,1,2,3,28,7,2025,26.57,24.20,26.57,train
8,201939,0022500210,2025-11-11,GSW @ OKC,L,20,4,13,0.308,1,...,0,1,5,11,-23,2025,26.75,22.80,26.75,train
9,201939,0022500220,2025-11-12,GSW @ SAS,W,34,13,25,0.520,5,...,0,1,2,46,5,2025,25.00,21.80,25.00,train
